<a href="https://colab.research.google.com/github/DataEngr-Raushan/My_New_Project/blob/DataEngineer%2FPySpark_09_July_2026/PySpark_Regression_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# **Running Pyspark in Colab**

To run spark in Colab, we need to first install all the dependencies in Colab environment i.e. Apache Spark 2.3.2 with hadoop 2.7, Java 8 and Findspark to locate the spark in the system. The tools installation can be carried out inside the Jupyter Notebook of the Colab. One important note is that if you are new in Spark, it is better to avoid Spark 2.4.0 version since some people have already complained about its compatibility issue with python.
Follow the steps to install the dependencies:

In [ ]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://www-us.apache.org/dist/spark/spark-2.4.1/spark-2.4.1-bin-hadoop2.7.tgz
!tar xf spark-2.4.1-bin-hadoop2.7.tgz
!pip install -q findspark

# New section

Now that you installed Spark and Java in Colab, it is time to set the environment path which enables you to run Pyspark in your Colab environment. Set the location of Java and Spark by running the following code:

In [ ]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-2.3.2-bin-hadoop2.7"

Run a local spark session to test your installation:

In [ ]:
import findspark
findspark.init()
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()

Congrats! Your Colab is ready to run Pyspark. Let's build a simple Linear Regression model.

# Linear Regression Model


Linear Regression model is one the oldest and widely used machine learning approach which assumes a relationship between dependent and independent variables. For example, a modeler might want to predict the forecast of the rain based on the humidity ratio. Linear Regression consists of the best fitting line through the scattered points on the graph and the best fitting line is known as the regression line.

The goal of this exercise to predict the housing prices by the given features. Let's predict the prices of the Boston Housing dataset by considering MEDV as the output variable and all the other variables as input.

Download the dataset from [here](https://github.com/asifahmed90/pyspark-ML-in-Colab/blob/master/BostonHousing.csv) and keep it somewhere on your computer. Load the dataset into your Colab directory from your local system:

In [ ]:
from google.colab import files
files.upload()

Check the dataset is uploaded correctly in the system by the following command

In [ ]:
!ls

BostonHousing.csv  spark-2.3.2-bin-hadoop2.7
sample_data	   spark-2.3.2-bin-hadoop2.7.tgz


Now that we have uploaded the dataset, we can start analyzing.
For our linear regression model we need to import two modules from Pyspark i.e. Vector Assembler and Linear Regression. Vector Assembler is a transformer that assembles all the features into one vector from multiple columns that contain type double. We could have used StringIndexer if any of our columns contains string values to convert it into numeric values. Luckily, the BostonHousing dataset only contains double values, so we don't need to worry about StringIndexer for now.

In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression

dataset = spark.read.csv('BostonHousing.csv',inferSchema=True, header =True)

Notice that we used InferSchema inside read.csv mofule. InferSchema enables us to infer automatically different data types for each column.

Let us print look into the dataset to see the data types of each column:

In [ ]:
dataset.printSchema()

root
 |-- crim: double (nullable = true)
 |-- zn: double (nullable = true)
 |-- indus: double (nullable = true)
 |-- chas: integer (nullable = true)
 |-- nox: double (nullable = true)
 |-- rm: double (nullable = true)
 |-- age: double (nullable = true)
 |-- dis: double (nullable = true)
 |-- rad: integer (nullable = true)
 |-- tax: integer (nullable = true)
 |-- ptratio: double (nullable = true)
 |-- b: double (nullable = true)
 |-- lstat: double (nullable = true)
 |-- medv: double (nullable = true)



Next step is to convert all the features from different columns into a single column and let's call this new vector column as 'Attributes' in the outputCol.

In [ ]:
#Input all the features in one vector column
assembler = VectorAssembler(inputCols=['crim', 'zn', 'indus', 'chas', 'nox', 'rm', 'age', 'dis', 'rad', 'tax', 'ptratio', 'b', 'lstat'], outputCol = 'Attributes')

output = assembler.transform(dataset)

#Input vs Output
finalized_data = output.select("Attributes","medv")

finalized_data.show()

+--------------------+----+
|          Attributes|medv|
+--------------------+----+
|[0.00632,18.0,2.3...|24.0|
|[0.02731,0.0,7.07...|21.6|
|[0.02729,0.0,7.07...|34.7|
|[0.03237,0.0,2.18...|33.4|
|[0.06905,0.0,2.18...|36.2|
|[0.02985,0.0,2.18...|28.7|
|[0.08829,12.5,7.8...|22.9|
|[0.14455,12.5,7.8...|27.1|
|[0.21124,12.5,7.8...|16.5|
|[0.17004,12.5,7.8...|18.9|
|[0.22489,12.5,7.8...|15.0|
|[0.11747,12.5,7.8...|18.9|
|[0.09378,12.5,7.8...|21.7|
|[0.62976,0.0,8.14...|20.4|
|[0.63796,0.0,8.14...|18.2|
|[0.62739,0.0,8.14...|19.9|
|[1.05393,0.0,8.14...|23.1|
|[0.7842,0.0,8.14,...|17.5|
|[0.80271,0.0,8.14...|20.2|
|[0.7258,0.0,8.14,...|18.2|
+--------------------+----+
only showing top 20 rows



Here, 'Attributes' are in the input features from all the columns and 'medv' is the target column.
Next, we should split the training and testing data according to our dataset (0.8 and 0.2 in this case).

In [ ]:
#Split training and testing data
train_data,test_data = finalized_data.randomSplit([0.8,0.2])


regressor = LinearRegression(featuresCol = 'Attributes', labelCol = 'medv')

#Learn to fit the model from training set
regressor = regressor.fit(train_data)

#To predict the prices on testing set
pred = regressor.evaluate(test_data)

#Predict the model
pred.predictions.show()

+--------------------+----+------------------+
|          Attributes|medv|        prediction|
+--------------------+----+------------------+
|[0.01301,35.0,1.5...|32.7| 30.07670363535312|
|[0.01538,90.0,3.7...|44.0| 37.75244575519337|
|[0.01778,95.0,1.4...|32.9|30.596108327253294|
|[0.0187,85.0,4.15...|23.1|25.717620889129734|
|[0.01965,80.0,1.7...|20.1|19.992379582220035|
|[0.02729,0.0,7.07...|34.7|30.425294527192754|
|[0.03113,0.0,4.39...|17.5|16.330496893793097|
|[0.03237,0.0,2.18...|33.4|28.578543755284294|
|[0.03306,0.0,5.19...|20.6| 22.16010760013387|
|[0.03359,75.0,2.9...|34.9| 34.42265990782376|
|[0.03537,34.0,6.0...|22.0|28.784081950984906|
|[0.03584,80.0,3.3...|23.5| 30.77179427151925|
|[0.03738,0.0,5.19...|20.7| 21.65956978285279|
|[0.04297,52.5,5.3...|24.8|26.706348196385573|
|[0.0456,0.0,13.89...|23.3|26.369847201011538|
|[0.04684,0.0,3.41...|22.6|26.949731074397704|
|[0.04981,21.0,5.6...|23.4| 23.90871028835852|
|[0.05372,0.0,13.9...|27.1|27.156639422924407|
|[0.05425,0.0

We can also print the coefficient and intercept of the regression model by using the following command:

In [ ]:
#coefficient of the regression model
coeff = regressor.coefficients

#X and Y intercept
intr = regressor.intercept

print ("The coefficient of the model is : %a" %coeff)
print ("The Intercept of the model is : %f" %intr)


The coefficient of the model is : DenseVector([-0.1239, 0.056, 0.0205, 2.7283, -16.8634, 3.218, 0.0163, -1.4331, 0.3657, -0.0134, -0.9328, 0.0096, -0.6229])
The Intercept of the model is : 39.049826


# Basic Statistical Analysis

Once we are done with the basic linear regression operation, we can go a bit further and analyze our model statistically by importing RegressionEvaluator module from Pyspark.

In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator
eval = RegressionEvaluator(labelCol="medv", predictionCol="prediction", metricName="rmse")

# Root Mean Square Error
rmse = eval.evaluate(pred.predictions)
print("RMSE: %.3f" % rmse)

# Mean Square Error
mse = eval.evaluate(pred.predictions, {eval.metricName: "mse"})
print("MSE: %.3f" % mse)

# Mean Absolute Error
mae = eval.evaluate(pred.predictions, {eval.metricName: "mae"})
print("MAE: %.3f" % mae)

# r2 - coefficient of determination
r2 = eval.evaluate(pred.predictions, {eval.metricName: "r2"})
print("r2: %.3f" %r2)



RMSE: 4.703
MSE: 22.118
MAE: 3.457
r2: 0.738


# **PySpark**

In [1]:
!pip install pyspark py4j

In [5]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("test_pyspark").getOrCreate()

In [7]:
df= spark.read.csv("/content/Data/CSV/customer.csv", header=True, inferSchema=True)
df.show()

+--------------+--------------------+---------------+----------------+-----------------+--------------------+--------------------+-------------+--------+----------+---------+----------------------+-----------+
|customerNumber|        customerName|contactLastName|contactFirstName|            phone|        addressLine1|        addressLine2|         city|   state|postalCode|  country|salesRepEmployeeNumber|creditLimit|
+--------------+--------------------+---------------+----------------+-----------------+--------------------+--------------------+-------------+--------+----------+---------+----------------------+-----------+
|           103|   Atelier graphique|        Schmitt|         Carine |       40.32.2555|      54, rue Royale|                null|       Nantes|    null|     44000|   France|                  1370|      21000|
|           112|  Signal Gift Stores|           King|            Jean|       7025551838|     8489 Strong St.|                null|    Las Vegas|      NV|     83

In [8]:
from pyspark.sql import SparkSession

if __name__ == '__main__':
    spark = SparkSession.builder.master("local[2]").appName("pySpark Application").getOrCreate()
    sc = spark.sparkContext
    list1 = [1,1,3,4, 6,7,8,2,4,5,7,9,2,4,8,4]
    rdd_list1 = sc.parallelize(list1)
    # print(rdd_list1.collect())
    print(rdd_list1.take(5))
    print(rdd_list1.takeSample(False, 20, 1))
    rdd_list2 = rdd_list1.map(lambda x : x * 2).filter(lambda x: x > 2)
    print(rdd_list2.collect())
    print(rdd_list2.reduce(lambda acc,value: acc + value))

    spark.stop()

[1, 1, 3, 4, 6]
[3, 7, 1, 8, 8, 7, 4, 4, 2, 9, 4, 1, 2, 4, 5, 6]
[6, 8, 12, 14, 16, 4, 8, 10, 14, 18, 4, 8, 16, 8]
146


In [15]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.master("local[2]").appName("pySpark Application").getOrCreate()

sc = spark.sparkContext

ebookRDD = sc.textFile("/content/Data/TEXT/pg1342.txt")

ebookRDD1 = (ebookRDD
                 .filter(lambda x: x.strip() != "")
                 .map(lambda x: x.lower())
                 )
ebookRDD2 = (ebookRDD1
                 .flatMap(lambda x: x.split(" "))
                 .filter(lambda x: x.strip() != "")
                 )
ebookRDD3 = ebookRDD2.map(lambda x: (x,1))

ebookRDD4 = (ebookRDD3
                 .reduceByKey(lambda acc,value: acc + value)
                 .sortBy(lambda x: x[1], False)
                 # .filter(lambda x: x[1] < 150)
                 )

print(ebookRDD1.take(5))
print(ebookRDD2.take(15))
print(ebookRDD3.take(10))
print(ebookRDD4.take(10))


['the project gutenberg ebook of pride and prejudice', 'this ebook is for the use of anyone anywhere in the united states and', 'most other parts of the world at no cost and with almost no restrictions', 'whatsoever. you may copy it, give it away or re-use it under the terms', 'of the project gutenberg license included with this ebook or online']
['the', 'project', 'gutenberg', 'ebook', 'of', 'pride', 'and', 'prejudice', 'this', 'ebook', 'is', 'for', 'the', 'use', 'of']
[('the', 1), ('project', 1), ('gutenberg', 1), ('ebook', 1), ('of', 1), ('pride', 1), ('and', 1), ('prejudice', 1), ('this', 1), ('ebook', 1)]
[('the', 4801), ('to', 4325), ('of', 3917), ('and', 3534), ('a', 2060), ('in', 2002), ('her', 1988), ('was', 1819), ('i', 1764), ('she', 1645)]


In [16]:
ebookRDD4.saveAsTextFile("/content/Data/TEXT/OUTPUT/ebook")
spark.stop()

In [22]:
from pyspark.sql import SparkSession
if __name__ == '__main__':
  spark = SparkSession.builder.master("local[2]").appName("pySpark Application").getOrCreate()
  sc = spark.sparkContext

  df_order=spark.read.csv("/content/Data/CSV/order.csv", header=True, inferSchema=True)
  df_order.show()

  df_product=spark.read.csv("/content/Data/CSV/product.csv", header=True, inferSchema=True)
  df_product.show()

+-----------+----------+------------+-----------+-------+--------------------+--------------+
|orderNumber| orderDate|requiredDate|shippedDate| status|            comments|customerNumber|
+-----------+----------+------------+-----------+-------+--------------------+--------------+
|      10100|2003-01-06|  2003-01-13| 2003-01-10|Shipped|                null|           363|
|      10101|2003-01-09|  2003-01-18| 2003-01-11|Shipped|Check on availabi...|           128|
|      10102|2003-01-10|  2003-01-18| 2003-01-14|Shipped|                null|           181|
|      10103|2003-01-29|  2003-02-07| 2003-02-02|Shipped|                null|           121|
|      10104|2003-01-31|  2003-02-09| 2003-02-01|Shipped|                null|           141|
|      10105|2003-02-11|  2003-02-21| 2003-02-12|Shipped|                null|           145|
|      10106|2003-02-17|  2003-02-24| 2003-02-21|Shipped|                null|           278|
|      10107|2003-02-24|  2003-03-03| 2003-02-26|Shipped|Dif

In [17]:
from pyspark.sql import SparkSession

 #Creating a Function
def getRDDFromFileWithHeader(sparkcontext, fileLoc, recordSep = ","):
    inputRDD = sparkcontext.textFile(fileLoc)
    odheader = inputRDD.first()
    outputRDD = (inputRDD
                 .filter(lambda x: x != odheader)
                 .map(lambda x: x.split(recordSep)))
    return outputRDD

if __name__ == '__main__':
    spark = SparkSession.builder.master("local[2]").appName("pySpark Application").getOrCreate()
    sc = spark.sparkContext

    orderdetailFileLoc = "/content/Data/CSV/order.csv"
    productFileLoc = "/content/Data/CSV/product.csv"

    # orderdetailRDD = sc.textFile(orderdetailFileLoc)
    # odheader = orderdetailRDD.first()
    # orderdetailRDD1 = (orderdetailRDD
    #                    .filter(lambda x: x != odheader)
    #                    .map(lambda x: x.split(",")))

    # find the total sales amount = sum(qty * price) for each productcode
    orderdetailRDD = getRDDFromFileWithHeader(sc, orderdetailFileLoc)
    print(orderdetailRDD.take(5))
    orderdetailRDD1 = orderdetailRDD.map(lambda x : (x[1], int(x[2]) * float(x[3])))
    productSalesRDD = (orderdetailRDD1
                       .reduceByKey(lambda acc, value : acc + value)
                       .mapValues(lambda x: round(x))
                       )
    # print(orderdetailRDD1.take(5))
    print(productSalesRDD.take(5))

    productRDD = getRDDFromFileWithHeader(sc, productFileLoc)
    productPairRDD = productRDD.map(lambda x : (x[0], (x[1], x[2])))
    print(productPairRDD.take(5))
    productSummaryRDD = (productPairRDD
                         .join(productSalesRDD)
                         .map(lambda x : x[0] + "," + x[1][0][0] + "," + x[1][0][1] + "," + str(x[1][1]))
                         )
    productSummaryRDD.saveAsTextFile("/content/Data/CSV/OUTPUT/prodSummary")

    print(productSummaryRDD.take(5))




    spark.stop()

[['10100', '2003-01-06', '2003-01-13', '2003-01-10', 'Shipped', 'null', '363'], ['10101', '2003-01-09', '2003-01-18', '2003-01-11', 'Shipped', 'Check on availability.', '128'], ['10102', '2003-01-10', '2003-01-18', '2003-01-14', 'Shipped', 'null', '181'], ['10103', '2003-01-29', '2003-02-07', '2003-02-02', 'Shipped', 'null', '121'], ['10104', '2003-01-31', '2003-02-09', '2003-02-01', 'Shipped', 'null', '141']]


Py4JJavaError: An error occurred while calling z:org.apache.spark.api.python.PythonRDD.runJob.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 1 in stage 2.0 failed 1 times, most recent failure: Lost task 1.0 in stage 2.0 (TID 3) (29ad7fde8b6e executor driver): org.apache.spark.api.python.PythonException: Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pyspark/python/lib/pyspark.zip/pyspark/worker.py", line 2044, in main
    process()
  File "/usr/local/lib/python3.12/dist-packages/pyspark/python/lib/pyspark.zip/pyspark/worker.py", line 2034, in process
    out_iter = func(split_index, iterator)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pyspark/core/rdd.py", line 5306, in pipeline_func
    return func(split, prev_func(split, iterator))
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pyspark/core/rdd.py", line 5306, in pipeline_f

# Dataframe

In [17]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql.functions import *

# This function now correctly handles either a StructType for schema or a boolean for inferSchema
def getDataframeFromFileWithHeader(sparkSession, fileSchema, fileLoc, fieldSep=","):
   inputDF=(sparkSession
   .read
   .option("header", "true")
   .option("sep", fieldSep)
   .option("nullValue", "null") #converts 'null' string to Null Value
   .schema(fileSchema)
   .csv(fileLoc))
   return inputDF

# No 'if __name__ == "__main__":' block in Colab cells for direct execution
spark = SparkSession.builder.master("local[2]").appName("pySpark Application").getOrCreate()

orderFileLoc = "/content/order.csv"
orderdetailFileLoc = "/content/orderdetail.csv"
productFileLoc = "/content/product.csv"

orderdetailSchema = StructType([
    StructField("orderNumber", IntegerType(), True),
    StructField("productCode", StringType(), True),
    StructField("quantityOrdered", IntegerType(), True),
    StructField("priceEach", DoubleType(), True),
    StructField("orderLineNumber", IntegerType(), True)
])

orderSchema="orderNumber int, orderDate date, requiredDate date, shippedDate date, Status string, comments string, customerNumber int"

orderDF=getDataframeFromFileWithHeader(spark, orderSchema, orderFileLoc)
# orderDF.show()


# Pass orderdetailSchema as the second argument to getDataframeFromFileWithHeader

# orderdetailDF = getDataframeFromFileWithHeader(spark, orderdetailSchema, orderdetailFileLoc)
# orderdetailDF.show()
# orderdetailDF.printSchema()

# orderDF.select("orderNumber","orderDate","status").where(col("status")
# !="cancelled").show()

# orderDF.na.fill("NA", subset=["status","comments"]).show() #Fill null vallue with NA on selected columns ( if not selected, would applied on the DataFrame)

print(orderDF.na.drop(how="any", subset=["orderDate","requiredDate","shippedDate"]).count()) #here, 'any' would behave as a 'or' between the given columns,data is null for the selected columns, it will drop the 'row' not 'column'.

print(orderDF.na.drop(how="all", subset=["orderDate","requiredDate","shippedDate"]).count()) #here, 'all' would behave as a 'and' between the given columns, data is null for the selected columns, it will drop the 'row' not 'column'.



# Stop the Spark session when done to release resources
spark.stop()

312
326
